# Saint-Venant pure bending of a prismatic beam — 3D solid benchmark

First validation case for genuine **3D solid elasticity** (3 parametric directions,
`BSplineVolume`) in the `future` module, assembled with classical Gauss quadrature
(`PatchIntegrator`'s 3-basis constructor + `Elasticity3D`).

A prismatic beam of length $L$ and rectangular cross-section $H \times B$ is bent by a
pure moment $M$ about the $z$-axis — no transverse load, no body force. The closed-form
displacement and stress fields are classical (Timoshenko & Goodier, *Theory of
Elasticity*, 3rd ed., §20, "Pure Bending of Prismatical Bars"):

$$
u_x = C\,x\,y, \qquad
u_y = -\tfrac{C}{2}\!\left(x^2 + \nu y^2 - \nu z^2\right), \qquad
u_z = -C\,\nu\,y\,z,
\qquad C = \frac{M}{E I_z},\quad I_z = \frac{B H^3}{12}
$$

$$
\sigma_{xx} = \frac{M y}{I_z}, \qquad \text{all other stress components} = 0.
$$

**Boundary conditions**

| Face | BC type | Condition |
|---|---|---|
| $x=0$ | Dirichlet — nonzero lift | exact field $\mathbf{u}$ (see below) |
| $x=L$ | Neumann — Python callable | exact traction $t_x = M y / I_z$ |
| $y=\pm H/2$, $z=\pm B/2$ | Natural (traction-free) | $\boldsymbol{\sigma}\cdot\mathbf{n} = \mathbf{0}$ |

**Material**: full 3D isotropic elasticity (`Elasticity3D`), $E = 210000$, $\nu = 0.3$.

This case was chosen specifically because it needs **zero body force** — no volume-load
assembler exists in this codebase. The $x=0$ Dirichlet face uses a genuinely **nonzero**
lift ($u = u_D + u_0$), new relative to every earlier 2D notebook (which only ever used
homogeneous or symmetry boundary conditions).

**What exactly is prescribed at $x=0$?** Not a plain clamp ($\mathbf{u}=\mathbf{0}$) —
the restriction of the exact field above to that face:
$$
u_x|_{x=0} = 0, \qquad
u_y|_{x=0} = -\tfrac{C}{2}\nu\left(y^2 - z^2\right), \qquad
u_z|_{x=0} = -C\,\nu\,y\,z.
$$
$u_x$ is zero (it is $C x y$, trivially zero at $x=0$), but $u_y$ and $u_z$ are **not** —
they are the classical Poisson/*anticlastic-curvature* warping of the cross-section
(bending on one side, Poisson bulging on the other, quadratic in $y,z$). A plain clamp
would be incompatible with the exact Saint-Venant solution and would introduce a local
boundary layer at $x=0$ — defeating the point of the exact-reproduction check below,
which needs the computed solution to match the exact field *everywhere*, including at
the Dirichlet face itself.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla

from yeti_iga.future.bspline import (
    BSpline, BSplineVolume, ControlPointManager,
    Patch, PRefiner, SubdivisionRefiner,
    GlobalDOFManager, PatchDOFManager,
    PatchIntegrator, PatchEvaluator, Material, Elasticity3D, IGABasis1D,
    Traction3D, ScalarLocalOperator3D,
)

## Geometry — minimal affine `BSplineVolume` box

The beam occupies $x \in [0, L]$, $y \in [-H/2, H/2]$, $z \in [-B/2, B/2]$. Starting from
a single-element, degree-1 box (8 control points at the corners) keeps the geometry
map **affine by construction** — exactly representable at any degree after k-refinement,
since degree elevation and knot insertion reproduce an affine function exactly (this is
the whole point of k-refinement: it only ever *adds* representable functions to the
space, never loses what was already there).

In [ ]:
L, H, B = 4.0, 1.0, 1.0   # beam length, cross-section height (y), width (z)
E, NU = 210000.0, 0.3
M_BEND = 1.0              # applied bending moment about z
IZ = B * H**3 / 12.0

kv0 = np.array([0., 0., 1., 1.])
su0, sv0, sw0 = BSpline(1, kv0), BSpline(1, kv0), BSpline(1, kv0)
tensor0 = BSplineVolume(su0, sv0, sw0)

mgr0 = ControlPointManager(dim=3)
for iw in range(2):
    for iv in range(2):
        for iu in range(2):
            x = iu * L
            y = (iv - 0.5) * H
            z = (iw - 0.5) * B
            mgr0.add_point([x, y, z])

# DOF manager attached from the start (3 scalar DOFs per CP): PRefiner/HRefiner
# (and SubdivisionRefiner, which delegates to HRefiner) keep an already-attached
# PatchDOFManager in sync automatically, rebuilding it in place to track the
# refined control net -- see the next section.
gdm = GlobalDOFManager([3] * 8)
pdm = PatchDOFManager(3, list(range(8)), gdm)
patch = Patch(tensor0, mgr0, list(range(8)), [2, 2, 2], pdm)
print(f'Initial mesh (degree 1, 1 element/direction): {patch.n_cp} control points')

In [ ]:
def plot_control_net_3d(patch, ax=None, title=None):
    """Minimal 3D scatter of a patch's control points (no dedicated 3D-solid
    plotting helper exists yet in postprocessing/ -- unlike plot_patches_2d()
    for 2D patches)."""
    coords = np.array([patch.control_point(i) for i in range(patch.n_cp)])
    if ax is None:
        fig = plt.figure(figsize=(5, 5))
        ax = fig.add_subplot(projection='3d')
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c='C0', s=15)
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_box_aspect([L, H, B])
    if title:
        ax.set_title(title)
    return ax

plot_control_net_3d(patch, title=f'Initial control net ({patch.n_cp} CPs)')
plt.show()

## k-refinement

Degree elevation (`PRefiner`) followed by uniform knot insertion (`SubdivisionRefiner`),
applied independently in each of the 3 parametric directions — both refiners already
loop generically over `tensor.components.size()`, so no 3D-specific code was needed for
this step; it is exercised here exactly as it already is for 2D patches.

Target: degree 2 (needed to represent the quadratic $u_y$ field exactly), 2 elements per
direction.

In [ ]:
degree_target, nbel_target = 2, 2

for d in range(3):
    if degree_target > 1:
        PRefiner(direction=d, n_elevations=degree_target - 1).refine(patch)

n_levels = nbel_target.bit_length() - 1
if n_levels > 0:
    for d in range(3):
        SubdivisionRefiner(direction=d, n_levels=n_levels).refine(patch)

print(f'After k-refinement: degree {[c.degree for c in patch.tensor.components]}, '
      f'local_shape {patch.local_shape}, {patch.n_cp} CPs')
plot_control_net_3d(patch, title=f'After k-refinement ({patch.n_cp} CPs)')
plt.show()

## Stiffness matrix

Three scalar DOFs ($u_x$, $u_y$, $u_z$) per control point. Since a `PatchDOFManager` was
already attached before k-refinement, `patch.dof_manager` is already up to date after
refining — `PRefiner`/`HRefiner` (and `SubdivisionRefiner`, which delegates to
`HRefiner`) detect an attached DOF manager and rebuild it in place to track the refined
control net, so there is no need for a separate "refine geometry-only, then attach DOFs"
step for this single, isolated patch (that two-stage pattern, used in every 2D notebook
in this codebase, exists for multi-patch assemblies, where global DOF numbering must
wait until *every* patch has reached its final refined size).

`Elasticity3D` is full 3D isotropic elasticity (no plane-stress/strain reduction) and
`PatchIntegrator`'s 4th constructor overload (patch, basis_u, basis_v, basis_w, law)
dispatches to the 3D solid stiffness kernel.

In [ ]:
n_cp = patch.n_cp
n_dof = 3 * n_cp

su, sv, sw = patch.tensor.components
basis_u = IGABasis1D.build(su, su.degree + 1)
basis_v = IGABasis1D.build(sv, sv.degree + 1)
basis_w = IGABasis1D.build(sw, sw.degree + 1)

law = Elasticity3D(Material(E=E, nu=NU))
integrator = PatchIntegrator(patch, basis_u, basis_v, basis_w, law)
K = integrator.integrate_stiffness()

print(f'n_cp = {n_cp}, n_dof = {n_dof}')
print(f'K: {K.shape}, nnz = {K.nnz}, symmetry error = {np.max(np.abs(K - K.T)):.2e}')

## Python-callable `Traction3D` — exact bending load on the $x=L$ face

A 3D boundary load integrates over a **face** (2 varying parametric directions), so the
surface element is $dA = |T_p \times T_q|$ (cross product of the two physical tangent
vectors), unlike the 2D case's curve-tangent norm $ds=|T|$ — this is exercised here via
`integrate_boundary_load_3d(direction=0, side=1, traction)` (the $x=L$ face).

Pure bending has **zero net force** on the loaded face (the resultant of $\sigma_{xx}$
over a symmetric cross-section vanishes since it is odd in $y$) — only a net moment.

In [ ]:
class BendingTraction(Traction3D):
    """Exact Saint-Venant pure-bending traction on the x=L face: t = (M*y/Iz, 0, 0)."""

    def __init__(self, M, Iz):
        super().__init__()
        self.M = M
        self.Iz = Iz

    def evaluate(self, pt):
        _, y, _ = pt
        return np.array([self.M * y / self.Iz, 0.0, 0.0])


F = integrator.integrate_boundary_load_3d(0, 1, BendingTraction(M_BEND, IZ))
print(f'F: {F.shape}, ||F|| = {np.linalg.norm(F):.6f}')
print(f'Net force on x=L face: Fx={F[0::3].sum():.3e}, Fy={F[1::3].sum():.3e}, '
      f'Fz={F[2::3].sum():.3e}  (all ~0: pure bending has zero net force)')

## Nonzero Dirichlet lift via Greville collocation

The $x=0$ face needs the exact field's own control-point coefficients, used both as the
Dirichlet data (the lift $u = u_D + u_0$) and as the reference for the exact-reproduction
check below.

**A subtlety**: naively assigning each control point's Dirichlet value to the exact field
evaluated at its Greville-abscissae-mapped physical location is only exact for an
**affine** (degree-1) field. The pure-bending $u_y$ field is **quadratic**, so instead the
true control coefficients are recovered by solving a **collocation** linear system at the
Greville abscissae — classical, well-conditioned, and exact whenever the target function
truly lies in the spline space (Schoenberg–Whitney / de Boor), which this quadratic field
does at degree $\ge 2$. The per-direction 1D collocation matrices are solved by sum
factorization (one direction at a time), the same separation-of-variables idea used
throughout this codebase's matrix-free tensor contractions.

In [ ]:
C_BEND = M_BEND / (E * IZ)


def exact_u(x, y, z):
    ux = C_BEND * x * y
    uy = -0.5 * C_BEND * (x**2 + NU * y**2 - NU * z**2)
    uz = -C_BEND * NU * y * z
    return np.array([ux, uy, uz])


def exact_grad_u(x, y, z):
    # grad[i, j] = d(u_i) / d(x_j), (x_j) = (x, y, z)
    return np.array([
        [C_BEND * y, C_BEND * x, 0.0],
        [-C_BEND * x, -C_BEND * NU * y, C_BEND * NU * z],
        [0.0, -C_BEND * NU * z, -C_BEND * NU * y],
    ])


def greville_abscissae(spline):
    kv = spline.knot_vector
    p = spline.degree
    n = len(kv) - p - 1
    return np.array([np.mean(kv[i + 1:i + p + 1]) for i in range(n)])


def collocation_matrix_1d(spline, gr):
    """C[i, a] = N_a(greville_i); solving C @ d = f gives the true control
    coefficients of f whenever f lies in the spline space."""
    n = len(gr)
    Cmat = np.zeros((n, n))
    for i in range(n):
        for a in range(n):
            Cmat[i, a] = spline.one_basis_fun(gr[i], a)
    return Cmat


def fit_exact_field(components, grs, phys_map, field_fn, n_components):
    """Exact B-spline control coefficients of an analytic vector field via
    per-direction collocation solves (sum factorization), u-fastest
    Fortran-order flattening."""
    Cs = [collocation_matrix_1d(components[d], grs[d]) for d in range(3)]
    n_u, n_v, n_w = len(grs[0]), len(grs[1]), len(grs[2])

    Fgrid = np.zeros((n_u, n_v, n_w, n_components))
    for iw in range(n_w):
        for iv in range(n_v):
            for iu in range(n_u):
                xyz = phys_map(grs[0][iu], grs[1][iv], grs[2][iw])
                Fgrid[iu, iv, iw, :] = field_fn(*xyz)

    D = Fgrid
    for c in range(n_components):
        d = D[:, :, :, c]
        d = np.linalg.solve(Cs[0], d.reshape(n_u, -1)).reshape(n_u, n_v, n_w)
        d = np.linalg.solve(Cs[1], d.transpose(1, 0, 2).reshape(n_v, -1)).reshape(n_v, n_u, n_w).transpose(1, 0, 2)
        d = np.linalg.solve(Cs[2], d.transpose(2, 0, 1).reshape(n_w, -1)).reshape(n_w, n_u, n_v).transpose(1, 2, 0)
        D[:, :, :, c] = d
    return D


grs = [greville_abscissae(c) for c in patch.tensor.components]


def phys_map(xi, eta, zeta):
    return xi * L, (eta - 0.5) * H, (zeta - 0.5) * B


D = fit_exact_field(patch.tensor.components, grs, phys_map, exact_u, 3)
u_exact_coeffs = np.zeros(n_dof)
u_exact_coeffs[0::3] = D[:, :, :, 0].flatten(order='F')
u_exact_coeffs[1::3] = D[:, :, :, 1].flatten(order='F')
u_exact_coeffs[2::3] = D[:, :, :, 2].flatten(order='F')
print('Exact control coefficients fitted.')

## Boundary conditions and solve

Standard nonzero-lift technique: prescribe $u_0$ on the fixed DOFs (the $x=0$ face),
move their contribution to the right-hand side, and solve the free sub-system —
$K_{ff}\,u_{0,f} = F_f - K_{f,fixed}\,u_{D,fixed}$.

In [ ]:
fixed_cp = np.array(sorted(patch.boundary_control_points(0, 0)))   # x=0 face
fixed_dofs = np.sort(np.concatenate([3*fixed_cp, 3*fixed_cp+1, 3*fixed_cp+2]))
free_dofs = np.setdiff1d(np.arange(n_dof), fixed_dofs)

u_c_full = np.zeros(n_dof)
u_c_full[fixed_dofs] = u_exact_coeffs[fixed_dofs]

K_csr = K.tocsr()
rhs_free = F[free_dofs] - (K_csr @ u_c_full)[free_dofs]
K_ff = K_csr[free_dofs][:, free_dofs].tocsc()

u_sol = np.zeros(n_dof)
u_sol[fixed_dofs] = u_exact_coeffs[fixed_dofs]
u_sol[free_dofs] = spla.spsolve(K_ff, rhs_free)

print(f'Solve done. ||u|| = {np.linalg.norm(u_sol):.6f}')
print(f'Constrained DOFs: {len(fixed_dofs)}, free DOFs: {len(free_dofs)}')

## Exact reproduction check

At degree $\ge 2$, the quadratic pure-bending field is exactly representable by the FE
space, so the computed solution must match the exact control coefficients fitted above
to near machine precision — the cheapest, strongest possible correctness check: any bug
in the 3D Jacobian, the stiffness kernel, the boundary-load surface Jacobian, or the DOF
scatter would show up here as an $O(1)$ discrepancy, not a small numerical error.

$L^2$/$H^1$-seminorm errors are computed via `ScalarLocalOperator3D`: a Python subclass
providing the integrand at one Gauss point, while `PatchIntegrator` drives the 3D Gauss
loop, the Jacobian, and the physical-space gradients.

In [ ]:
max_abs_diff = np.max(np.abs(u_sol - u_exact_coeffs))
print(f'max|u_h - u_exact| = {max_abs_diff:.3e} (machine-precision exact reproduction expected)')


class L2ErrorOperator(ScalarLocalOperator3D):
    """Integrand |u_h - u_ex|^2 for the L2 error norm."""

    def compute_scalar_integrand(self, R, dRdx, dRdy, dRdz, pt, u_local):
        R = np.asarray(R)
        u = np.asarray(u_local)
        uh = np.array([R @ u[0::3], R @ u[1::3], R @ u[2::3]])
        ex = exact_u(*pt)
        return float(np.sum((uh - ex) ** 2))


class H1SemiNormErrorOperator(ScalarLocalOperator3D):
    """Integrand |grad(u_h) - grad(u_ex)|^2 for the H1 semi-norm error."""

    def compute_scalar_integrand(self, R, dRdx, dRdy, dRdz, pt, u_local):
        dRdx, dRdy, dRdz = np.asarray(dRdx), np.asarray(dRdy), np.asarray(dRdz)
        u = np.asarray(u_local)
        ux, uy, uz = u[0::3], u[1::3], u[2::3]
        grad_h = np.array([
            [dRdx @ ux, dRdy @ ux, dRdz @ ux],
            [dRdx @ uy, dRdy @ uy, dRdz @ uy],
            [dRdx @ uz, dRdy @ uz, dRdz @ uz],
        ])
        grad_ex = exact_grad_u(*pt)
        return float(np.sum((grad_h - grad_ex) ** 2))


l2_err = np.sqrt(integrator.integrate_scalar_operator_3d(L2ErrorOperator(), u_sol))
h1_err = np.sqrt(integrator.integrate_scalar_operator_3d(H1SemiNormErrorOperator(), u_sol))
print(f'L2 error = {l2_err:.3e}')
print(f'H1 semi-norm error = {h1_err:.3e}')

## Visualisation

Undeformed vs. exaggerated deformed control net, and a displacement-magnitude slice at
$z=0$ (exact vs. FE, evaluated via `PatchEvaluator` — already fully N-D generic,
dispatching on `patch.dof_manager.dofs_per_control_point` and the parametric dimension
count at runtime, so no changes were needed for 3D use).

In [ ]:
scale = 0.2 / np.max(np.abs(u_sol))   # exaggeration factor for a visible deformed shape
coords0 = np.array([patch.control_point(i) for i in range(n_cp)])
u_per_cp = u_sol.reshape(-1, 3)
coords_def = coords0 + scale * u_per_cp

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(projection='3d')
ax.scatter(*coords0.T, c='C0', s=10, alpha=0.4, label='undeformed')
ax.scatter(*coords_def.T, c='C3', s=10, label=f'deformed (x{scale:.1f})')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
ax.set_box_aspect([L, H, B])
ax.legend()
ax.set_title('Pure bending: undeformed vs. exaggerated deformed control net')
plt.show()

In [ ]:
n_pts = 40
u_par = np.linspace(0.01, 0.99, n_pts)
v_par = np.linspace(0.01, 0.99, n_pts)
UU, VV = np.meshgrid(u_par, v_par)
params = np.column_stack([UU.ravel(), VV.ravel(), np.full(UU.size, 0.5)])

evaluator = PatchEvaluator(patch)
u_h = evaluator.evaluate_solution(params, u_sol)
u_h_mag = np.linalg.norm(u_h, axis=1)

phys_pts = np.array([phys_map(u, v, 0.5) for u, v, _ in params])
u_ex_mag = np.array([np.linalg.norm(exact_u(*p)) for p in phys_pts])

X = phys_pts[:, 0].reshape(n_pts, n_pts)
Y = phys_pts[:, 1].reshape(n_pts, n_pts)
U_ex = u_ex_mag.reshape(n_pts, n_pts)
U_h = u_h_mag.reshape(n_pts, n_pts)

vmin, vmax = min(U_ex.min(), U_h.min()), max(U_ex.max(), U_h.max())
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
c0 = axes[0].pcolormesh(X, Y, U_ex, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
axes[0].set_title('Exact $|u|$ (z=0 slice)'); plt.colorbar(c0, ax=axes[0])
c1 = axes[1].pcolormesh(X, Y, U_h, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
axes[1].set_title('FE $|u_h|$ (z=0 slice)'); plt.colorbar(c1, ax=axes[1])
c2 = axes[2].pcolormesh(X, Y, np.abs(U_h - U_ex), cmap='Reds', shading='auto')
axes[2].set_title('Pointwise error'); plt.colorbar(c2, ax=axes[2])
for ax in axes:
    ax.set_aspect('equal'); ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout()
plt.show()

## Convergence study — h-refinement (degree 1)

Degree 1 cannot represent the quadratic bending field exactly, so refining the mesh
(1/2/4/8/16 elements per direction, via `SubdivisionRefiner`) should show the standard
a priori FE error estimates: $L^2 \sim \mathcal{O}(h^2)$, $H^1$ semi-norm $\sim
\mathcal{O}(h^1)$.

**Note**: the measured $H^1$ rate exceeds the nominal order-1 estimate here (approaching
~1.6) — this is **Gauss-point superconvergence**, a known, benign phenomenon on this kind
of structured, affine mesh with a polynomial exact solution (the error norm is evaluated
at the same Gauss points used for assembly). It is not a correctness concern: the
exact-reproduction check above already confirms correctness to machine precision
independently of this convergence study.

In [ ]:
def build_beam_patch_deg1(nbel):
    kv = np.array([0., 0., 1., 1.])
    splines = [BSpline(1, kv) for _ in range(3)]
    tensor = BSplineVolume(*splines)
    mgr = ControlPointManager(dim=3)
    for iw in range(2):
        for iv in range(2):
            for iu in range(2):
                mgr.add_point([iu * L, (iv - 0.5) * H, (iw - 0.5) * B])

    gdm_ = GlobalDOFManager([3] * 8)
    pdm_ = PatchDOFManager(3, list(range(8)), gdm_)
    p = Patch(tensor, mgr, list(range(8)), [2, 2, 2], pdm_)   # DOF manager attached upfront

    n_levels = nbel.bit_length() - 1
    if n_levels > 0:
        for d in range(3):
            SubdivisionRefiner(direction=d, n_levels=n_levels).refine(p)
    return p


def run_pure_bending_deg1(nbel):
    p = build_beam_patch_deg1(nbel)
    n_dof_ = 3 * p.n_cp

    bu = IGABasis1D.build(p.tensor.components[0], 2)
    bv = IGABasis1D.build(p.tensor.components[1], 2)
    bw = IGABasis1D.build(p.tensor.components[2], 2)
    integ = PatchIntegrator(p, bu, bv, bw, Elasticity3D(Material(E=E, nu=NU)))

    K_ = integ.integrate_stiffness().tocsr()
    F_ = integ.integrate_boundary_load_3d(0, 1, BendingTraction(M_BEND, IZ))

    grs_ = [greville_abscissae(c) for c in p.tensor.components]
    D_ = fit_exact_field(p.tensor.components, grs_, phys_map, exact_u, 3)
    u_ex = np.zeros(n_dof_)
    u_ex[0::3] = D_[:, :, :, 0].flatten(order='F')
    u_ex[1::3] = D_[:, :, :, 1].flatten(order='F')
    u_ex[2::3] = D_[:, :, :, 2].flatten(order='F')

    fixed = np.array(sorted(p.boundary_control_points(0, 0)))
    fdofs = np.sort(np.concatenate([3*fixed, 3*fixed+1, 3*fixed+2]))
    free = np.setdiff1d(np.arange(n_dof_), fdofs)

    uc = np.zeros(n_dof_)
    uc[fdofs] = u_ex[fdofs]
    rhs = F_[free] - (K_ @ uc)[free]
    Kff = K_[free][:, free].tocsc()

    usol = np.zeros(n_dof_)
    usol[fdofs] = u_ex[fdofs]
    usol[free] = spla.spsolve(Kff, rhs)

    l2 = np.sqrt(integ.integrate_scalar_operator_3d(L2ErrorOperator(), usol))
    h1 = np.sqrt(integ.integrate_scalar_operator_3d(H1SemiNormErrorOperator(), usol))
    return l2, h1


nbels = [1, 2, 4, 8, 16]
l2_errs, h1_errs = [], []
for nbel in nbels:
    l2, h1 = run_pure_bending_deg1(nbel)
    l2_errs.append(l2)
    h1_errs.append(h1)
    print(f'nbel={nbel:2d}: L2={l2:.4e}, H1 semi-norm={h1:.4e}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.loglog(nbels, l2_errs, 'o-', label='$L^2$ error')
ax.loglog(nbels, h1_errs, 's-', label='$H^1$ semi-norm')

n_ref = np.array([nbels[0], nbels[-1]], dtype=float)
c2 = l2_errs[-1] / n_ref[-1] ** (-2)
c1 = h1_errs[-1] / n_ref[-1] ** (-1)
ax.loglog(n_ref, c2 * n_ref ** (-2), 'k--', alpha=0.4, label='slope -2 ($O(h^2)$)')
ax.loglog(n_ref, c1 * n_ref ** (-1), 'k:', alpha=0.4, label='slope -1 ($O(h^1)$)')

ax.set_xlabel('Elements per direction')
ax.set_ylabel('Error')
ax.set_title('Pure bending h-convergence (degree 1)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()